# 04 — AutoCalibrate: Full Calibration Pipeline

`AutoCalibrate` runs the complete ge-transition calibration automatically:

```
res_spec → qubit_spec → power_rabi → t1 → ramsey → spin_echo → ss_opt
```

After each step it:
1. Updates `ExperimentConfig` in-place so later steps see the new values
2. Persists the result in `CalibrationStore` (timestamped JSON on disk)

You can **skip** any step with `skip=(...)` and resume just the steps you need.

In [1]:
import sys; sys.path.insert(0, '../')
import tempfile, os

from QickworkspaceV2 import BaseExperiment, CalibrationStore, AutoCalibrate, ExperimentConfig
from QickworkspaceV2.config.system_cfg import config_list

BaseExperiment.connect_pyro4(
    ns_host='192.168.10.82', ns_port=8888, proxy_name='myqick',
    data_path=r'D:\Labber_Data\Jay\test',
)

qubit = 'Q1'
cfg_all = ExperimentConfig(config_list)
store_path = os.path.join(tempfile.gettempdir(), 'autocal_demo.json')
store = CalibrationStore(store_path, default_max_age_hours=24)


c:\Users\cluster\anaconda3\envs\qick_gui\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
QICK library version mismatch: 0.2.381 remote (the board), 0.2.405 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
myqick PYRO:obj_a0fa10b880d442359a82595a954ab5e1@192.168.10.82:44291
[BaseExperiment] Session activated: QICK@192.168.10.82:8888/myqick, data_path='D:\\Labber_Data\\Jay\\test'


## Run the full pipeline

In [2]:
auto = AutoCalibrate(cfg_all, qubit=qubit, cal_store=store)

# Run current pipeline steps. Bad fits can still raise RuntimeError on unconnected/invalid samples.
auto.run(skip=('spin_echo', 'ss_opt'))


Traceback (most recent call last):
  File "c:\Users\cluster\anaconda3\envs\qick_gui\Lib\site-packages\IPython\core\interactiveshell.py", line 3579, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\cluster\AppData\Local\Temp\ipykernel_18924\640717577.py", line 4, in <module>
    auto.run(skip=('spin_echo', 'ss_opt'))
  File "c:\Users\cluster\Desktop\QickworksaceV2_proj-test2\tutorial\..\QickworkspaceV2\calibration\pipeline.py", line 170, in run
    fn()
  File "c:\Users\cluster\Desktop\QickworksaceV2_proj-test2\tutorial\..\QickworkspaceV2\calibration\pipeline.py", line 261, in step_power_rabi
    self._require_result("power_rabi", result, required=("pi_gain", "pi2_gain"))
  File "c:\Users\cluster\Desktop\QickworksaceV2_proj-test2\tutorial\..\QickworkspaceV2\calibration\pipeline.py", line 105, in _require_result
    raise RuntimeError(f"{step}: analysis failed: {result.quality_message}")
RuntimeError: power_rabi: analysis failed: pi_gain=1.012 out of [0.

## Inspect the results

In [3]:
# AutoCalibrate.results stores calibrated parameter values keyed by config/result name.
print('=== AutoCalibrate results ===')
for key, value in auto.results.items():
    print(f'  {key:<28} {value}')


=== AutoCalibrate results ===
  res_freq_ge                  6723.3997
  qb_freq_ge                   2847.459


In [4]:
# CalibrationStore — what was persisted
print('=== Calibration Store ===')
print(store.summary('Q1'))

=== Calibration Store ===
[Q1]
  res_freq_ge                  = 6723.3997  @ 2026-08-03T21:20:01.819076
  qb_freq_ge                   = 2847.459  @ 2026-08-03T21:20:04.120653
  qb_mixer                     = 2847.459  @ 2026-08-03T21:20:04.121615
  sigma_ge                     = 0.05  @ 2026-08-03T21:20:04.122613


In [5]:
# Live config — updated in-place by AutoCalibrate
live = cfg_all.get_qubit('Q1')
print('=== Live ExperimentConfig (Q1) ===')
for key in ('res_freq_ge', 'qb_freq_ge', 'pi_gain_ge'):
    print(f'  {key:<20} = {live[key]}')

=== Live ExperimentConfig (Q1) ===
  res_freq_ge          = 6723.3997
  qb_freq_ge           = 2847.459
  pi_gain_ge           = 0.1


## Running only stale steps

A common pattern at the start of a lab session: check which parameters have gone stale
and only run those steps.

In [6]:
# Map step names to the parameter they calibrate
STEP_PARAM = {
    'res_spec':   ('res_freq_ge',  48),
    'qubit_spec': ('qb_freq_ge',   12),
    'power_rabi': ('pi_gain_ge',   12),
    'ramsey':     ('qb_freq_ge',    6),
    't1':         ('T1_us',        24),
}

skip = []
run  = []
for step, (param, max_age_h) in STEP_PARAM.items():
    if store.is_stale('Q1', param, max_age_hours=max_age_h):
        run.append(step)
    else:
        skip.append(step)

print('Will run :', run)
print('Will skip:', skip)

# Then:
# auto2 = AutoCalibrate(cfg_all, 'Q1', cal_store=store)
# auto2.run(skip=tuple(skip))

Will run : ['power_rabi', 't1']
Will skip: ['res_spec', 'qubit_spec', 'ramsey']


## Persist to disk — reload on next session

In [7]:
# The store auto-saves after every set().  Reload is transparent:
store2 = CalibrationStore(store_path)
print('Reloaded qb_freq_ge:', store2.get('Q1', 'qb_freq_ge'))

# Populate ExperimentConfig from the persisted store
flat = store2.to_flat_dict('Q1')
for k, v in flat.items():
    try:
        cfg_all.update(k, v, q_index='Q1')
    except Exception:
        pass   # keys like T1_us not in config are fine to skip

print('Live qb_freq_ge after reload:', cfg_all.get_qubit('Q1')['qb_freq_ge'])

Reloaded qb_freq_ge: 2847.459
Live qb_freq_ge after reload: 2847.459


**Next:** [05_custom_experiment.ipynb](05_custom_experiment.ipynb) — writing your own experiment class.